In [1]:
# Set project root
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

while PROJECT_ROOT.name != "archivist" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

if PROJECT_ROOT.name != "archivist":
    raise RuntimeError("Could not find Archivist project root")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: /home/kali/workspace/github.com/tlazizbek/archivist


In [2]:
# Import libraries
import json
import sqlite3
import pandas as pd
import numpy as np

In [3]:
# Use the correct database
DB_PATH = PROJECT_ROOT / "archivist.db"

print("Database:", DB_PATH)
print("Exists:", DB_PATH.exists())

if not DB_PATH.exists():
    raise FileNotFoundError(f"Database not found: {DB_PATH}")

Database: /home/kali/workspace/github.com/tlazizbek/archivist/archivist.db
Exists: True


In [4]:
with sqlite3.connect(DB_PATH) as connection:
    tables = pd.read_sql_query(
        """
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
        ORDER BY name
        """,
        connection,
    )

tables

,name
0,chunks
1,documents
2,query_feedback
3,query_logs
4,sqlite_sequence


In [5]:
with sqlite3.connect(DB_PATH) as connection:
    query_logs = pd.read_sql_query(
        """
        SELECT
            id,
            query_text,
            retrieval_method,
            retrieved_chunk_ids,
            answer_text,
            latency_ms,
            llm_model,
            created_at
        FROM query_logs
        ORDER BY id
        """,
        connection,
    )

print("Rows:", len(query_logs))
query_logs.head()

Rows: 52


,id,query_text,retrieval_method,retrieved_chunk_ids,answer_text,latency_ms,llm_model,created_at
0,1,What is the main topic of these documents?,keyword,"[547, 544, 561, 155, 372]","Based on the provided context, there is no sin...",12804,openrouter/free,2026-08-26 09:18:17
1,2,What teaching methods are discussed in the doc...,keyword,"[544, 542, 545, 549, 547]",User Safety: safe,6608,openrouter/free,2026-08-26 09:34:08
2,3,What role does reading play in teaching history?,keyword,"[545, 542, 544, 549, 541]",I do not have enough information.,60209,openrouter/free,2026-08-26 09:35:19
3,4,What are the characteristics of sperm whales?,keyword,"[122, 232, 233, 229, 385]","Based on the provided contexts, the following ...",23096,openrouter/free,2026-08-26 09:35:53
4,5,What does Moby-Dick say about whaling?,keyword,"[142, 153, 143, 451, 221]",I do not have enough information to answer the...,2587,openrouter/free,2026-08-26 09:36:06


In [6]:
with sqlite3.connect(DB_PATH) as connection:
    corpus_stats = pd.read_sql_query(
        """
        SELECT
            d.id AS document_id,
            d.title,
            d.ingested_at,
            COUNT(c.id) AS chunk_count,
            COALESCE(SUM(LENGTH(c.content)), 0) AS character_count
        FROM documents d
        LEFT JOIN chunks c
            ON c.document_id = d.id
        GROUP BY d.id, d.title, d.ingested_at
        ORDER BY d.id
        """,
        connection,
    )

print("Corpus stats:", corpus_stats.shape)

corpus_stats.head()

Corpus stats: (43, 5)


,document_id,title,ingested_at,chunk_count,character_count
0,1,pg2701,2026-08-05 12:00:44,480,1370075
1,2,pg79448,2026-08-05 09:31:44,106,308511
2,3,pg79443,2026-08-05 19:14:44,71,238135
3,4,pg1342,2026-08-06 03:57:44,290,818806
4,5,pg79447,2026-08-07 12:43:44,68,208389


In [7]:
# Corpus growth over time (source for the corpus-growth line chart)
corpus_stats["ingested_at"] = pd.to_datetime(corpus_stats["ingested_at"])

corpus_growth = (
    corpus_stats
    .assign(date=corpus_stats["ingested_at"].dt.date)
    .groupby("date")
    .agg(
        documents_added=("document_id", "count"),
        chunks_added=("chunk_count", "sum"),
    )
    .reset_index()
    .sort_values("date")
)

corpus_growth["cumulative_documents"] = corpus_growth["documents_added"].cumsum()
corpus_growth["cumulative_chunks"] = corpus_growth["chunks_added"].cumsum()

corpus_growth

,date,documents_added,chunks_added,cumulative_documents,cumulative_chunks
0,2026-08-05,3,657,3,657
1,2026-08-06,1,290,4,947
2,2026-08-07,2,379,6,1326
3,2026-08-08,3,302,9,1628
4,2026-08-09,2,652,11,2280
5,2026-08-10,2,207,13,2487
6,2026-08-11,1,358,14,2845
7,2026-08-12,3,622,17,3467
8,2026-08-13,2,213,19,3680
9,2026-08-14,1,112,20,3792


In [8]:
query_logs["created_at"] = pd.to_datetime(query_logs["created_at"])

queries_per_day = (
    query_logs
    .assign(date=query_logs["created_at"].dt.date)
    .groupby("date")
    .size()
    .reset_index(name="query_count")
)

queries_per_day

,date,query_count
0,2026-08-05,2
1,2026-08-06,1
2,2026-08-07,2
3,2026-08-08,1
4,2026-08-10,1
5,2026-08-11,2
6,2026-08-12,3
7,2026-08-14,1
8,2026-08-15,3
9,2026-08-17,1


In [9]:
average_latency_ms = query_logs["latency_ms"].mean()

print(f"Average latency: {average_latency_ms:.2f} ms")

Average latency: 20166.08 ms


In [10]:
p95_latency_ms = query_logs["latency_ms"].quantile(0.95)

print(f"P95 latency: {p95_latency_ms:.2f} ms")

P95 latency: 60815.60 ms


In [11]:
retrieval_method_usage = (
    query_logs["retrieval_method"]
    .value_counts()
    .rename_axis("retrieval_method")
    .reset_index(name="query_count")
)

retrieval_method_usage["percentage"] = (
    retrieval_method_usage["query_count"]
    / retrieval_method_usage["query_count"].sum()
    * 100
)

retrieval_method_usage

,retrieval_method,query_count,percentage
0,hybrid,23,44.230769
1,semantic,15,28.846154
2,keyword,14,26.923077


In [12]:
query_logs["retrieved_chunk_ids"] = query_logs["retrieved_chunk_ids"].apply(
    lambda value: json.loads(value) if isinstance(value, str) else value
)

query_logs[["id", "retrieved_chunk_ids"]]

,id,retrieved_chunk_ids
0,1,"[547, 544, 561, 155, 372]"
1,2,"[544, 542, 545, 549, 547]"
2,3,"[545, 542, 544, 549, 541]"
3,4,"[122, 232, 233, 229, 385]"
4,5,"[142, 153, 143, 451, 221]"
5,6,"[542, 545, 549, 544, 541]"
6,7,"[122, 385, 387, 233, 333]"
7,8,"[545, 542, 544, 543, 522]"
8,9,"[285, 294, 117, 8, 181]"
9,10,"[542, 547, 545, 546, 548]"


In [13]:
retrieved_chunks = (
    query_logs[
        ["id", "retrieved_chunk_ids"]
    ]
    .explode("retrieved_chunk_ids")
    .rename(
        columns={
            "id": "query_id",
            "retrieved_chunk_ids": "chunk_id",
        }
    )
)

retrieved_chunks["chunk_id"] = retrieved_chunks["chunk_id"].astype(int)

retrieved_chunks.head(10)

,query_id,chunk_id
0,1,547
0,1,544
0,1,561
0,1,155
0,1,372
1,2,544
1,2,542
1,2,545
1,2,549
1,2,547


In [14]:
with sqlite3.connect(DB_PATH) as connection:
    chunk_documents = pd.read_sql_query(
        """
        SELECT
            c.id AS chunk_id,
            c.document_id,
            d.title
        FROM chunks c
        JOIN documents d
            ON d.id = c.document_id
        """,
        connection,
    )

chunk_documents.head()

,chunk_id,document_id,title
0,1,1,pg2701
1,2,1,pg2701
2,3,1,pg2701
3,4,1,pg2701
4,5,1,pg2701


In [15]:
retrieved_documents = retrieved_chunks.merge(
    chunk_documents,
    on="chunk_id",
    how="left",
)

most_retrieved_documents = (
    retrieved_documents
    .groupby(["document_id", "title"])
    .size()
    .reset_index(name="retrieval_count")
    .sort_values("retrieval_count", ascending=False)
    .reset_index(drop=True)
)

most_retrieved_documents

,document_id,title,retrieval_count
0,2,pg79448,65
1,1,pg2701,64
2,16,pg768,10
3,13,pg36,10
4,36,pg35,10
5,38,pg2554,10
6,43,pg215,10
7,24,pg84,10
8,6,pg79444,6
9,33,pg120,6


In [16]:
powerbi_queries = query_logs[
    [
        "id",
        "query_text",
        "retrieval_method",
        "latency_ms",
        "llm_model",
        "created_at",
    ]
].copy()

powerbi_queries["date"] = powerbi_queries["created_at"].dt.date

powerbi_queries

,id,query_text,retrieval_method,latency_ms,llm_model,created_at,date
0,1,What is the main topic of these documents?,keyword,12804,openrouter/free,2026-08-26 09:18:17,2026-08-26
1,2,What teaching methods are discussed in the doc...,keyword,6608,openrouter/free,2026-08-26 09:34:08,2026-08-26
2,3,What role does reading play in teaching history?,keyword,60209,openrouter/free,2026-08-26 09:35:19,2026-08-26
3,4,What are the characteristics of sperm whales?,keyword,23096,openrouter/free,2026-08-26 09:35:53,2026-08-26
4,5,What does Moby-Dick say about whaling?,keyword,2587,openrouter/free,2026-08-26 09:36:06,2026-08-26
5,6,What are primary sources in history education?,keyword,18469,openrouter/free,2026-08-26 09:36:31,2026-08-26
6,7,What superstitions about whales are mentioned?,keyword,7505,openrouter/free,2026-08-26 09:36:45,2026-08-26
7,8,How is history taught in schools and universit...,keyword,8225,openrouter/free,2026-08-26 09:37:00,2026-08-26
8,9,What information is given about sperm whale be...,keyword,30561,openrouter/free,2026-08-26 09:37:34,2026-08-26
9,10,How should educators approach historical subje...,semantic,77095,openrouter/free,2026-08-26 10:26:47,2026-08-26


In [17]:
powerbi_documents = most_retrieved_documents.copy()

powerbi_documents

,document_id,title,retrieval_count
0,2,pg79448,65
1,1,pg2701,64
2,16,pg768,10
3,13,pg36,10
4,36,pg35,10
5,38,pg2554,10
6,43,pg215,10
7,24,pg84,10
8,6,pg79444,6
9,33,pg120,6


In [18]:
OUTPUT_DIR = PROJECT_ROOT / "analytics" / "exports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

queries_csv = OUTPUT_DIR / "query_analytics.csv"
documents_csv = OUTPUT_DIR / "document_retrievals.csv"
corpus_growth_csv = OUTPUT_DIR / "corpus_growth.csv"
corpus_stats_csv = OUTPUT_DIR / "corpus_stats.csv"

powerbi_queries.to_csv(queries_csv, index=False)
powerbi_documents.to_csv(documents_csv, index=False)
corpus_growth.to_csv(corpus_growth_csv, index=False)
corpus_stats.to_csv(corpus_stats_csv, index=False)

print("Exported:")
print(queries_csv)
print(documents_csv)
print(corpus_growth_csv)
print(corpus_stats_csv)

Exported:
/home/kali/workspace/github.com/tlazizbek/archivist/analytics/exports/query_analytics.csv
/home/kali/workspace/github.com/tlazizbek/archivist/analytics/exports/document_retrievals.csv
/home/kali/workspace/github.com/tlazizbek/archivist/analytics/exports/corpus_growth.csv
/home/kali/workspace/github.com/tlazizbek/archivist/analytics/exports/corpus_stats.csv


In [19]:
print("Query analytics:")
print(pd.read_csv(queries_csv).head())

print("\nDocument retrievals:")
print(pd.read_csv(documents_csv).head())

print("\nCorpus growth:")
print(pd.read_csv(corpus_growth_csv).head())

Query analytics:
   id                                         query_text retrieval_method  \
0   1         What is the main topic of these documents?          keyword   
1   2  What teaching methods are discussed in the doc...          keyword   
2   3   What role does reading play in teaching history?          keyword   
3   4      What are the characteristics of sperm whales?          keyword   
4   5             What does Moby-Dick say about whaling?          keyword   

   latency_ms        llm_model           created_at        date  
0       12804  openrouter/free  2026-08-26 09:18:17  2026-08-26  
1        6608  openrouter/free  2026-08-26 09:34:08  2026-08-26  
2       60209  openrouter/free  2026-08-26 09:35:19  2026-08-26  
3       23096  openrouter/free  2026-08-26 09:35:53  2026-08-26  
4        2587  openrouter/free  2026-08-26 09:36:06  2026-08-26  

Document retrievals:
   document_id    title  retrieval_count
0            2  pg79448               65
1            1   pg2

In [20]:
assert not query_logs.empty, "query_logs is empty"
assert not corpus_stats.empty, "corpus_stats is empty"
assert not queries_per_day.empty, "queries_per_day is empty"
assert not retrieval_method_usage.empty, "retrieval_method_usage is empty"
assert not most_retrieved_documents.empty, "most_retrieved_documents is empty"
assert not corpus_growth.empty, "corpus_growth is empty"
assert queries_csv.exists(), "Query CSV was not created"
assert documents_csv.exists(), "Document CSV was not created"
assert corpus_growth_csv.exists(), "Corpus growth CSV was not created"

print("Day 21 analysis completed successfully.")
print()
print(f"Queries: {len(query_logs)}")
print(f"Documents: {len(corpus_stats)}")
print(f"Average latency: {average_latency_ms:.2f} ms")
print(f"P95 latency: {p95_latency_ms:.2f} ms")
print(f"Query CSV: {queries_csv}")
print(f"Document CSV: {documents_csv}")
print(f"Corpus growth CSV: {corpus_growth_csv}")

Day 21 analysis completed successfully.

Queries: 52
Documents: 43
Average latency: 20166.08 ms
P95 latency: 60815.60 ms
Query CSV: /home/kali/workspace/github.com/tlazizbek/archivist/analytics/exports/query_analytics.csv
Document CSV: /home/kali/workspace/github.com/tlazizbek/archivist/analytics/exports/document_retrievals.csv
Corpus growth CSV: /home/kali/workspace/github.com/tlazizbek/archivist/analytics/exports/corpus_growth.csv
